# TSS pairs analysis

This notebook characterizes the structural and expression properties of core and covariant transcription start site (TSS) pairs derived from the CoPTR clustering pipeline. Analysis is performed across four pair sets, defined by all pairwise combinations of input cohort (non-cancer / cancer) and the model used for latent vector extraction (non-cancer model / cancer model), as summarized in Table 1.

Model–data combinations used to define exclusive TSS pair sets.

| Symbol | Model         | Data          |
|--------|---------------|---------------|
| N•N    | Non-cancer    | Non-cancer    |
| N•C    | Non-cancer    | Cancer        |
| C•C    | Cancer        | Cancer        |
| C•N    | Cancer        | Non-cancer    |

For each combination, exclusive TSS pairs were extracted according to the protocol described in the Methods. 
The resulting pair sets (NN_TSS_PAIRS, NC_TSS_PAIRS, CC_TSS_PAIRS, CN_TSS_PAIRS) are provided as Supplementary Tables S5–S8.

CURATED_CAGE_PEAKS, NORMAL_CAGE_DERIVATIVES, and CANCER_CAGE_DERIVATIVES were generated using the framework established by Satish et al. (2025, unpublished; DOI: [10.64898/2025.12.22.696121](https://doi.org/10.64898/2025.12.22.696121)) and are not distributed within this repository.

## Main

In [2]:
# -- Standard library ----------------------------------------------------------
import numpy as np
import pandas as pd

# -- Statistics ----------------------------------------------------------------
from scipy.stats import spearmanr

# -- Plotting ------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
CURATED_CAGE_PEAKS       = None
NORMAL_CAGE_DERIVATIVES  = None
CANCER_CAGE_DERIVATIVES  = None

# -- Paths to TSS pair sets (accessible as supplementary) --------------------
NN_TSS_PAIRS = None
NC_TSS_PAIRS = None
CC_TSS_PAIRS = None
CN_TSS_PAIRS = None

In [ ]:
CAGE_PEAK_DATA = pd.read_csv(CURATED_CAGE_PEAKS, sep="\t", header=0)

NORMAL_CAGE_DERIVATIVES['e'] = (
    NORMAL_CAGE_DERIVATIVES['TCI_N'] * NORMAL_CAGE_DERIVATIVES['distance']
)
CANCER_CAGE_DERIVATIVES['e'] = (
    CANCER_CAGE_DERIVATIVES['TCI_C'] * CANCER_CAGE_DERIVATIVES['distance']
)

In [ ]:
# -- Load TSS pair sets --------------------------------------------------------
nn_tss_pairs = pd.read_csv(NN_TSS_PAIRS, header=0, index_col=None, sep="\t")
nc_tss_pairs = pd.read_csv(NC_TSS_PAIRS, header=0, index_col=None, sep="\t")
cc_tss_pairs = pd.read_csv(CC_TSS_PAIRS, header=0, index_col=None, sep="\t")
cn_tss_pairs = pd.read_csv(CN_TSS_PAIRS, header=0, index_col=None, sep="\t")

print(f"N-N pairs: {len(nn_tss_pairs):,}")
print(f"N-C pairs: {len(nc_tss_pairs):,}")
print(f"C-C pairs: {len(cc_tss_pairs):,}")
print(f"C-N pairs: {len(cn_tss_pairs):,}")

N-N pairs: 122,169
N-C pairs: 1,230,360
C-C pairs: 53,438
C-N pairs: 145,627


In [ ]:
# Extract unique core and covariant TSS identifiers for each pair set
nn_core_tss      = set(nn_tss_pairs['Core-TSS'])
nn_covariant_tss = set(nn_tss_pairs['Covariant-TSS'])

nc_core_tss      = set(nc_tss_pairs['Core-TSS'])
nc_covariant_tss = set(nc_tss_pairs['Covariant-TSS'])

cc_core_tss      = set(cc_tss_pairs['Core-TSS'])
cc_covariant_tss = set(cc_tss_pairs['Covariant-TSS'])

cn_core_tss      = set(cn_tss_pairs['Core-TSS'])
cn_covariant_tss = set(cn_tss_pairs['Covariant-TSS'])

## Helper Functions

In [ ]:
def fetch_core_covariant_interactions(
    tss_list: list,
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    For a given set of core TSSs, returns the number of unique covariant
    partners each core TSS has in the provided pair DataFrame.

    Parameters
    ----------
    tss_list : list
        Core TSS identifiers to query.
    df : pd.DataFrame
        TSS pair DataFrame with columns: Core-TSS, Covariant-TSS, Covariance.

    Returns
    -------
    pd.DataFrame
        One row per core TSS with column Unique-Covariant-TSS-Count,
        sorted descending by partner count.
    """
    mask = df["Core-TSS"].isin(tss_list)
    subset_df = df[mask]

    interaction_df = subset_df.groupby("Core-TSS").agg({
        "Covariant-TSS": [
            ('Unique-Count', 'nunique'),
            ('Labels', lambda x: list(set(x)))
        ],
        'Covariance': [
            ('Covariances', lambda x: list(set(x)))
        ]
    })

    interaction_df.columns = [
        "Unique-Covariant-TSS-Count",
        "Covariant-TSS-Labels",
        "Covariances"
    ]
    interaction_df = interaction_df.reset_index()

    return (
        interaction_df
        .sort_values(by='Unique-Covariant-TSS-Count', ascending=False)
        .reset_index(drop=True)
        .drop(columns=["Covariant-TSS-Labels", "Covariances"])
    )

## Covariant TSS Count per Core TSS

For each core TSS, we quantify the number of unique covariant partners across all four model•data combinations (N•N, N•C, C•C, C•N). This distribution reflects the connectivity of core TSSs within the co-expression landscape and provides insight into the extent to which individual core TSSs participate in broader regulatory modules. Core TSSs with a high covariant partner count may represent transcriptional hubs, while those with few partners suggest more context-specific or isolated regulatory activity (Figure 2I).

In [ ]:
nn_core_interactions = fetch_core_covariant_interactions(nn_core_tss, nn_tss_pairs)
nc_core_interactions = fetch_core_covariant_interactions(nc_core_tss, nc_tss_pairs)
cc_core_interactions = fetch_core_covariant_interactions(cc_core_tss, cc_tss_pairs)
cn_core_interactions = fetch_core_covariant_interactions(cn_core_tss, cn_tss_pairs)

## Comparison between Core *d*-rank vs Covariant sum-rank

Core TSSs are ranked by their *d* (distance between TSS and most upstreak start codon) and binned into percentiles. The total number of covariant partners is then summed per percentile bin. Spearman correlation is computed between distance-rank percentile and covariant-sum rank to test whether more distal core TSSs tend to have more covariant partners (Figure 2L and S1A). 

In [ ]:
def core_d_rank_vs_covar_sum_rank(tss_list, data_df, data):
    """
    Rank core TSSs by distance-to-ATG, bin into percentiles, sum covariant
    partner counts per bin, and compute Spearman correlation between
    distance-rank percentile and covariant-sum rank.

    Parameters
    ----------
    tss_list : set
        Core TSS identifiers.
    data_df : pd.DataFrame
        TSS pair DataFrame (Core-TSS, Covariant-TSS, Covariance).
    data : str
        Label for this combination (e.g. 'N-N').

    Returns
    -------
    ranked_df : pd.DataFrame
    spearman_stats : SpearmanrResult
    """
    core_interaction_df = fetch_core_covariant_interactions(
        tss_list=tss_list,
        df=data_df
    )

    core_interaction_df = core_interaction_df[['Core-TSS', 'Unique-Covariant-TSS-Count']]

    core_interaction_df = core_interaction_df.merge(
        right=CAGE_PEAK_DATA.set_index('p.gene')['distance'].rename('Core-d'),
        left_on='Core-TSS',
        right_index=True
    )

    core_interaction_df = core_interaction_df.sort_values(by='Core-d', ascending=False)
    core_interaction_df['Core-d-rank'] = pd.factorize(core_interaction_df['Core-d'])[0] + 1

    ranked_df = (
        core_interaction_df
        .groupby('Core-d-rank')['Unique-Covariant-TSS-Count']
        .agg('sum')
        .reset_index()
    )
    ranked_df['Core-d-rank-percentile'] = pd.qcut(
        ranked_df['Core-d-rank'], q=100, labels=False
    ) + 1

    ranked_df = (
        ranked_df
        .groupby('Core-d-rank-percentile')['Unique-Covariant-TSS-Count']
        .agg('sum')
        .reset_index()
    )

    ranked_df = ranked_df.sort_values(by='Unique-Covariant-TSS-Count', ascending=False)
    ranked_df['Covariant-sum-rank'] = pd.factorize(ranked_df['Unique-Covariant-TSS-Count'])[0] + 1
    ranked_df['data'] = data

    spearman_stats = spearmanr(ranked_df['Core-d-rank-percentile'], ranked_df['Covariant-sum-rank'])

    return ranked_df, spearman_stats

In [ ]:
nn_cov_sum_rank, nn_cov_sum_stats = core_d_rank_vs_covar_sum_rank(nn_core_tss, nn_tss_pairs, 'N-N')
nc_cov_sum_rank, nc_cov_sum_stats = core_d_rank_vs_covar_sum_rank(nc_core_tss, nc_tss_pairs, 'N-C')

cc_cov_sum_rank, cc_cov_sum_stats = core_d_rank_vs_covar_sum_rank(cc_core_tss, cc_tss_pairs, 'C-C')
cn_cov_sum_rank, cn_cov_sum_stats = core_d_rank_vs_covar_sum_rank(cn_core_tss, cn_tss_pairs, 'C-N')

print(f"N-N  rho={nn_cov_sum_stats.statistic:.4f}  p={nn_cov_sum_stats.pvalue:.2e}")
print(f"N-C  rho={nc_cov_sum_stats.statistic:.4f}  p={nc_cov_sum_stats.pvalue:.2e}")
print(f"C-C  rho={cc_cov_sum_stats.statistic:.4f}  p={cc_cov_sum_stats.pvalue:.2e}")
print(f"C-N  rho={cn_cov_sum_stats.statistic:.4f}  p={cn_cov_sum_stats.pvalue:.2e}")

## Core *d*-rank vs Core *E* sum-rank
Energy cost coefficient (*E*) = *TCI* × *d* is used as a composite activity measure per TSS. Core TSSsare ranked by *d*, binned into percentiles, and the sum of *E* per percentile bin is ranked. Spearman correlation is computed between the two rankseries to assess whether more distal TSSs also have higher *d* scores (Figure 3B and S2B).

In [ ]:
def core_d_rank_vs_core_e_sum_rank(tss_excl_list, cage_derivatives, data):
    """
    Rank core TSSs by distance-to-ATG, bin into percentiles, sum E per bin,
    and compute Spearman correlation between distance-rank percentile and
    E-sum rank.

    Parameters
    ----------
    tss_excl_list : set
        Core TSS identifiers.
    cage_derivatives : pd.DataFrame
        CAGE derivative table containing 'p.gene' and 'e' columns.
    data : str
        Label for this combination (e.g. 'N-N').

    Returns
    -------
    rank_df : pd.DataFrame
    spearman_stats : SpearmanrResult
    """
    data_df = (
        CAGE_PEAK_DATA
        .loc[CAGE_PEAK_DATA['p.gene'].isin(tss_excl_list)]
        .drop(columns=['chr', 'start', 'end', 'strand'])
        .reset_index(drop=True)
    )

    data_df = data_df.merge(
        right=cage_derivatives[['p.gene', 'e']].rename(columns={'e': 'Core-e'}),
        on='p.gene'
    )

    data_df = data_df.sort_values(by='distance', ascending=False)
    data_df['Core-d-rank'] = pd.factorize(data_df['distance'])[0] + 1

    rank_df = data_df.groupby('Core-d-rank')['Core-e'].agg('sum').reset_index()

    rank_df['Core-d-rank-percentile'] = pd.qcut(
        rank_df['Core-d-rank'], q=100, labels=False
    ) + 1

    rank_df = rank_df.groupby('Core-d-rank-percentile')['Core-e'].agg('sum').reset_index()

    rank_df = rank_df.sort_values(by='Core-e', ascending=False)
    rank_df['Core-e-rank'] = pd.factorize(rank_df['Core-e'])[0] + 1
    rank_df['data'] = data

    spearman_stats = spearmanr(rank_df['Core-d-rank-percentile'], rank_df['Core-e-rank'])

    return rank_df, spearman_stats

In [ ]:
nn_e_sum_ranked, nn_e_sum_stats = core_d_rank_vs_core_e_sum_rank(nn_core_tss, NORMAL_CAGE_DERIVATIVES, 'N-N')
nc_e_sum_ranked, nc_e_sum_stats = core_d_rank_vs_core_e_sum_rank(nc_core_tss, CANCER_CAGE_DERIVATIVES, 'N-C')

cc_e_sum_ranked, cc_e_sum_stats = core_d_rank_vs_core_e_sum_rank(cc_core_tss, CANCER_CAGE_DERIVATIVES, 'C-C')
cn_e_sum_ranked, cn_e_sum_stats = core_d_rank_vs_core_e_sum_rank(cn_core_tss, NORMAL_CAGE_DERIVATIVES, 'C-N')

print(f"N-N  rho={nn_e_sum_stats.statistic:.4f}  p={nn_e_sum_stats.pvalue:.2e}")
print(f"N-C  rho={nc_e_sum_stats.statistic:.4f}  p={nc_e_sum_stats.pvalue:.2e}")
print(f"C-C  rho={cc_e_sum_stats.statistic:.4f}  p={cc_e_sum_stats.pvalue:.2e}")
print(f"C-N  rho={cn_e_sum_stats.statistic:.4f}  p={cn_e_sum_stats.pvalue:.2e}")

## Covariant count-rank vs Core *E* sum-rank

Core TSSs are ranked by the number of covariant partners (connectivity rank). *E* values are summed per connectivity-rank percentile bin and ranked. Spearman correlation is computed to test whether highly connected core TSSs also have higher cumulative expression-distance scores (Figure 3C and S1C).

In [ ]:
def core_rank_vs_core_e_sum_rank(tss_list, data_df, cage_derivatives, data):
    """
    Rank core TSSs by covariant partner count, bin into percentiles, sum E
    per bin, and compute Spearman correlation between connectivity-rank
    percentile and E-sum rank.

    Parameters
    ----------
    tss_list : set
        Core TSS identifiers.
    data_df : pd.DataFrame
        TSS pair DataFrame (Core-TSS, Covariant-TSS, Covariance).
    cage_derivatives : pd.DataFrame
        CAGE derivative table containing 'p.gene' and 'e' columns.
    data : str
        Label for this combination (e.g. 'N-N').

    Returns
    -------
    ranked_df : pd.DataFrame
    spearman_stats : SpearmanrResult
    """
    core_interaction_df = fetch_core_covariant_interactions(
        tss_list=tss_list,
        df=data_df
    )

    core_interaction_df = core_interaction_df.sort_values(
        by='Unique-Covariant-TSS-Count', ascending=False
    )
    core_interaction_df['Core-rank'] = (
        pd.factorize(core_interaction_df['Unique-Covariant-TSS-Count'])[0] + 1
    )

    core_interaction_df = core_interaction_df.merge(
        right=cage_derivatives.set_index('p.gene')[['e']].rename(columns={'e': 'Core-e'}),
        left_on='Core-TSS',
        right_index=True
    )

    ranked_df = core_interaction_df.groupby('Core-rank')['Core-e'].agg('sum').reset_index()

    ranked_df['Core-rank-percentile'] = pd.qcut(
        ranked_df['Core-rank'], q=100, labels=False
    ) + 1

    ranked_df = ranked_df.groupby('Core-rank-percentile')['Core-e'].agg('sum').reset_index()
    ranked_df = ranked_df.sort_values(by='Core-e', ascending=False)
    ranked_df['Core-e-rank'] = pd.factorize(ranked_df['Core-e'])[0] + 1
    ranked_df['data'] = data

    spearman_stats = spearmanr(ranked_df['Core-rank-percentile'], ranked_df['Core-e-rank'])

    return ranked_df, spearman_stats

In [ ]:
nn_e_cov_ranked, nn_e_cov_stats = core_rank_vs_core_e_sum_rank(nn_core_tss, nn_tss_pairs, NORMAL_CAGE_DERIVATIVES, 'N-N')
nc_e_cov_ranked, nc_e_cov_stats = core_rank_vs_core_e_sum_rank(nc_core_tss, nc_tss_pairs, CANCER_CAGE_DERIVATIVES, 'N-C')

cc_e_cov_ranked, cc_e_cov_stats = core_rank_vs_core_e_sum_rank(cc_core_tss, cc_tss_pairs, CANCER_CAGE_DERIVATIVES, 'C-C')
cn_e_cov_ranked, cn_e_cov_stats = core_rank_vs_core_e_sum_rank(cn_core_tss, cn_tss_pairs, NORMAL_CAGE_DERIVATIVES, 'C-N')

print(f"N-N  rho={nn_e_cov_stats.statistic:.4f}  p={nn_e_cov_stats.pvalue:.2e}")
print(f"N-C  rho={nc_e_cov_stats.statistic:.4f}  p={nc_e_cov_stats.pvalue:.2e}")
print(f"C-C  rho={cc_e_cov_stats.statistic:.4f}  p={cc_e_cov_stats.pvalue:.2e}")
print(f"C-N  rho={cn_e_cov_stats.statistic:.4f}  p={cn_e_cov_stats.pvalue:.2e}")

## Core *d*-rank vs Mean core *E*-rank

For each core TSS, a normalised activity score (E / covariant partner count) is computed. Core TSSs are binned by distance-rank percentile and the mean normalised score per bin is ranked. Spearman correlation tests whether distance rank predicts mean normalised activity rank (Figure 3D and S1D). 

In [ ]:
def core_d_vs_core_mean_e(data_df, tss_list, cage_derivatives, data):
    """
    Compute per-TSS normalised activity (E / covariant count), bin by
    distance-rank percentile, and compute Spearman correlation between
    distance-rank percentile and mean normalised activity rank.

    Parameters
    ----------
    data_df : pd.DataFrame
        TSS pair DataFrame (Core-TSS, Covariant-TSS, Covariance).
    tss_list : set
        Core TSS identifiers.
    cage_derivatives : pd.DataFrame
        CAGE derivative table containing 'p.gene' and 'e' columns.
    data : str
        Label for this combination (e.g. 'N-N').

    Returns
    -------
    rank_df : pd.DataFrame
    spearman_stats : SpearmanrResult
    """
    interaction_df = fetch_core_covariant_interactions(
        tss_list=tss_list,
        df=data_df
    )

    interaction_df = interaction_df[['Core-TSS', 'Unique-Covariant-TSS-Count']]

    interaction_df = interaction_df.merge(
        right=cage_derivatives.set_index('p.gene')['e'].rename('Core-e'),
        left_on='Core-TSS',
        right_index=True
    )

    interaction_df = interaction_df.merge(
        right=CAGE_PEAK_DATA.set_index('p.gene')['distance'].rename('Core-d'),
        left_on='Core-TSS',
        right_index=True
    )

    # Normalise E by covariant partner count
    interaction_df['Core-e/Cov-count'] = (
        interaction_df['Core-e'] / interaction_df['Unique-Covariant-TSS-Count']
    )

    interaction_df = interaction_df.sort_values(by='Core-d', ascending=False)
    interaction_df['Core-d-rank'] = pd.factorize(interaction_df['Core-d'])[0] + 1

    rank_df = interaction_df.groupby('Core-d-rank')['Core-e/Cov-count'].agg('mean').reset_index()

    rank_df['Core-d-rank-percentile'] = pd.qcut(
        rank_df['Core-d-rank'], q=100, labels=False
    ) + 1

    rank_df = rank_df.groupby('Core-d-rank-percentile')['Core-e/Cov-count'].agg('mean').reset_index()

    rank_df = rank_df.sort_values(by='Core-e/Cov-count', ascending=False)
    rank_df['Mean-e-rank'] = pd.factorize(rank_df['Core-e/Cov-count'])[0] + 1
    rank_df['data'] = data

    spearman_stats = spearmanr(rank_df['Core-d-rank-percentile'], rank_df['Mean-e-rank'])

    return rank_df, spearman_stats

In [ ]:
nn_mean_e_ranks, nn_mean_e_stats = core_d_vs_core_mean_e(nn_tss_pairs, nn_core_tss, NORMAL_CAGE_DERIVATIVES, 'N-N')
nc_mean_e_ranks, nc_mean_e_stats = core_d_vs_core_mean_e(nc_tss_pairs, nc_core_tss, CANCER_CAGE_DERIVATIVES, 'N-C')

cc_mean_e_ranks, cc_mean_e_stats = core_d_vs_core_mean_e(cc_tss_pairs, cc_core_tss, CANCER_CAGE_DERIVATIVES, 'C-C')
cn_mean_e_ranks, cn_mean_e_stats = core_d_vs_core_mean_e(cn_tss_pairs, cn_core_tss, NORMAL_CAGE_DERIVATIVES, 'C-N')

print(f"N-N  rho={nn_mean_e_stats.statistic:.4f}  p={nn_mean_e_stats.pvalue:.2e}")
print(f"N-C  rho={nc_mean_e_stats.statistic:.4f}  p={nc_mean_e_stats.pvalue:.2e}")
print(f"C-C  rho={cc_mean_e_stats.statistic:.4f}  p={cc_mean_e_stats.pvalue:.2e}")
print(f"C-N  rho={cn_mean_e_stats.statistic:.4f}  p={cn_mean_e_stats.pvalue:.2e}")

## Gini Coefficient and Lorenz Curve

The Gini coefficient quantifies inequality in the distribution of scaled CAGE scores (or E values) across TSSs. A Lorenz curve is plotted for each group. Values are computed after excluding zeros (unexpressed samples).

Two variants are visualised:
- **E** — Gini of the expression-distance product (TCI × distance)
- **Scaled score** — Gini of the raw scaled CAGE scores

$$G = \left(\frac{2\sum_{i=1}^{n} i \cdot x_i}{n\sum_{i=1}^{n} x_i}\right) - \frac{n+1}{n}$$

In [ ]:
def gini(x):
    """
    Compute the Gini coefficient of a 1D non-negative array.
    Array must be sorted ascending before calling.
    """
    n = len(x)
    cumulative_index = np.arange(1, n + 1)
    return (2 * np.sum(cumulative_index * x) / (n * np.sum(x))) - (n + 1) / n


def compute_lorenz(data):
    """
    Compute the Lorenz curve and Gini coefficient for a 1D array.

    Returns
    -------
    pop_proportions : np.ndarray  — x-axis (cumulative population share)
    lorenz_values   : np.ndarray  — y-axis (cumulative value share)
    gini_coeff      : float
    """
    data_sorted = np.sort(data)
    cumsum      = np.cumsum(data_sorted)
    total       = cumsum[-1]

    lorenz_values    = np.empty(len(data_sorted) + 1)
    lorenz_values[0] = 0
    lorenz_values[1:] = cumsum / total

    pop_proportions = np.linspace(0, 1, len(lorenz_values))
    gini_coeff      = gini(data_sorted)

    return pop_proportions, lorenz_values, gini_coeff


def calculate_gini_plot(
    data, tss_lists, labels, style_map,
    save=False,
    png_save_path='lorenz_curve.png',
    svg_save_path='lorenz_curve.svg'
):
    """
    Plot Lorenz curves and compute Gini coefficients for multiple TSS groups.

    Parameters
    ----------
    data          : pd.DataFrame — full CAGE score or E matrix (TSSs as index)
    tss_lists     : list of sets — TSS identifiers for each curve
    labels        : list of (group, curve_type) tuples matching style_map keys
                    e.g. [('N-N', 'Core'), ('N-N', 'Covariant'), ('N-C', 'Core')]
    style_map     : dict — visual style per group and curve type
    save          : bool — if True, saves the figure
    png_save_path : str  — output path for PNG
    svg_save_path : str  — output path for SVG
    """
    plt.figure(figsize=(10, 10))
    plt.plot([0, 1], [0, 1], label="Perfect Equality", color='lightgrey', linestyle='--')

    for i, (tss_list, (group, curve_type)) in enumerate(zip(tss_lists, labels), 1):
        style  = style_map[group][curve_type]
        color  = style['color']
        ls     = style['ls']
        alpha  = style['alpha']
        lw     = style['lw']
        zorder = style['zorder']

        print(f"\n--- Curve {i}: {group} | {curve_type} ---")
        df           = data[data.index.isin(tss_list)]
        flat         = df.to_numpy().ravel()
        flat_filtered = flat[flat != 0]   # exclude zeros (unexpressed samples)
        print(f"  {len(flat_filtered):,} non-zero values "
              f"({len(flat) - len(flat_filtered):,} zeros excluded)")

        pop_proportions, lorenz_values, gini_coeff = compute_lorenz(flat_filtered)
        print(f"  Gini = {gini_coeff:.4f}")

        plt.plot(
            pop_proportions, lorenz_values,
            label=f"{group} {curve_type} (Gini={gini_coeff:.4f})",
            color=color, linestyle=ls, alpha=alpha, lw=lw, zorder=zorder,
        )

    plt.xlabel("Cumulative proportion of TSSs", fontsize=21)
    plt.ylabel("Cumulative proportion of total scaled scores", fontsize=21)

    ax = plt.gca()
    ax.tick_params(axis='both', which='major', labelsize=22)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.legend(frameon=False, fontsize=15)
    plt.grid(True, linestyle=':', alpha=0.6)

    if save:
        plt.savefig(png_save_path, dpi=300, bbox_inches='tight')

    plt.show()
    print("\nDone.")

### Lorenz Curve: E (TCI x distance)

Scaled CAGE scores are multiplied by TSS-to-ATG distance to compute E.
The Lorenz curve is plotted for each group using `calculate_gini_plot` above.

### Lorenz Curve: Scaled CAGE Score

Raw scaled CAGE scores are used directly to compute the Gini coefficient and
Lorenz curve for each group using `calculate_gini_plot` above.